In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Sree Ram Roshan A S - 24BAD113

In [1]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Model name:", MODEL_NAME)

Device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model name: Qwen/Qwen2.5-0.5B-Instruct


In [2]:
messages = [
    {
        "role": "user",
        "content": "What is Artificial Intelligence?"
    }
]

# Tokenize the prompt
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# Handle either a tensor or a BatchEncoding safely
if isinstance(inputs, dict) or hasattr(inputs, "input_ids"):
    input_ids = inputs["input_ids"] if isinstance(inputs, dict) else inputs.input_ids
else:
    input_ids = inputs

# Move input IDs to the model device
input_ids = input_ids.to(device)

# Generate the response
with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens
response = tokenizer.decode(
    output_ids[0, input_ids.shape[1]:],
    skip_special_tokens=True
)

print("User: What is Artificial Intelligence?")
print("Chatbot:", response)

User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves developing algorithms and models that can process information, learn from data, and make decisions based on that learning.

Key features of AI include:

1. **Learning**: The ability to improve its performance over time through experience.
2. **Reasoning**: The capacity to understand and use language effectively.
3. **Perception**: The capability to perceive and respond to stimuli in


# Part B

In [3]:
# PART B: PROMPT HANDLING AND RESPONSE GENERATION (FIXED)

def generate_response(user_message):
    try:
        if not user_message or not user_message.strip():
            return "Please enter a valid message."

        messages = [
            {"role": "user", "content": user_message.strip()}
        ]

        # Convert prompt into a PyTorch tensor
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )

        # Extract the tensor if a BatchEncoding is returned
        if not isinstance(input_ids, torch.Tensor):
            input_ids = input_ids["input_ids"]

        input_ids = input_ids.to(device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            output_ids[0, input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        return response or "The model did not generate a response."

    except Exception as e:
        return f"Error generating response: {type(e).__name__}: {e}"


# Test the corrected function
print(generate_response("Explain the Transformer architecture simply."))

The Transformer architecture is a type of neural network model that has been widely used in natural language processing (NLP) and other areas of computer science. It consists of several key components:

1. **Encoder**: The encoder takes input sequences as inputs and produces representations of these sequences. This representation can be thought of as a summary or "feature vector" of the sequence.

2. **Decoder**: The decoder takes the encoded representations as inputs and generates output sequences based on the learned rules from the encoder. The decoder's goal is to produce outputs that are similar to the input sequences.

3. **Attention Mechanism**: Attention mechanisms are used to control how much information each part of the input sequence contributes to the final output. This helps in improving the quality


In [4]:
test_prompts = [
    "What is a Large Language Model?",
    "Explain the Transformer architecture in simple words.",
    "What is the purpose of a tokenizer?",
    "Write a Python program to add two numbers.",
    "Give three advantages of artificial intelligence."
]

for i, prompt in enumerate(test_prompts, start=1):
    print(f"\nTest {i}")
    print("User:", prompt)
    print("Chatbot:", generate_response(prompt))
    print("-" * 60)


Test 1
User: What is a Large Language Model?
Chatbot: A Large Language Model (LLM) refers to a type of artificial intelligence that can generate human-like text based on the input it receives. These models are designed to understand and respond to complex natural language questions or queries, often using large amounts of data and knowledge from various sources.

Key characteristics of LLMs include:

1. **Natural Language Processing**: The ability to process and interpret human language.
2. **Contextual Understanding**: Being able to grasp the context in which a question or statement is made.
3. **Knowledge Base**: Having access to vast amounts of information and domain-specific knowledge.
4. **Generative Power**: Generating coherent and meaningful responses to inputs.
5. **Relevance**: Responding with content relevant to the user's
------------------------------------------------------------

Test 2
User: Explain the Transformer architecture in simple words.
Chatbot: The Transformer 

In [5]:
print(generate_response(""))

Please enter a valid message.


# Part C

In [6]:
def chatbot_reply(message, history):
    try:
        if not message or not message.strip():
            return "Please enter a valid message."

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful AI chatbot. "
                    "Give clear, relevant, and concise answers."
                )
            }
        ]

        for item in (history or [])[-6:]:
            if isinstance(item, dict):
                role = item.get("role")
                content = item.get("content", "")

                if role in ("user", "assistant") and isinstance(content, str):
                    messages.append({
                        "role": role,
                        "content": content
                    })

        messages.append({
            "role": "user",
            "content": message.strip()
        })

        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            output_ids[0][input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        return response or "Sorry, I could not generate a response."

    except Exception as e:
        return f"An error occurred: {str(e)}"


demo = gr.ChatInterface(
    fn=chatbot_reply,
    title="Simple LLM Chatbot",
    description="A Transformer-based chatbot built using Hugging Face and Gradio.",
    textbox=gr.Textbox(
        placeholder="Type your message here...",
        label="Your Message"
    )
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57f4194d3dc36bb6d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Part D

In [7]:
import pandas as pd

evaluation_prompts = [
    ("Factual", "What is a Large Language Model?"),
    ("Factual", "What is the role of self-attention in Transformers?"),
    ("Conversational", "Hello! How are you?"),
    ("Conversational", "Can you explain that in simpler words?"),
    ("Task-oriented", "Write Python code to find the sum of two numbers."),
    ("Task-oriented", "List three applications of AI.")
]

results = []

for category, prompt in evaluation_prompts:
    answer = generate_response(prompt)

    results.append({
        "Category": category,
        "User Prompt": prompt,
        "Chatbot Response": answer
    })

    print("\nCategory:", category)
    print("User:", prompt)
    print("Chatbot:", answer)
    print("-" * 60)

results_df = pd.DataFrame(results)

results_df.to_csv("chatbot_test_results.csv", index=False)

print("\nTesting completed.")
print(results_df[["Category", "User Prompt"]])
print("\nResults saved to chatbot_test_results.csv")


Category: Factual
User: What is a Large Language Model?
Chatbot: A Large Language Model (LLM) refers to a type of artificial intelligence that can generate human-like text based on the input it receives. These models are designed to understand and respond to complex natural language questions or queries, often using large amounts of data and knowledge from various sources.

Key characteristics of LLMs include:

1. **Natural Language Processing**: The ability to process and interpret human language.
2. **Contextual Understanding**: Being able to grasp the context in which a question or statement is made.
3. **Knowledge Base**: Having access to vast amounts of information and domain-specific knowledge.
4. **Generative Power**: Generating coherent and meaningful responses to inputs.
5. **Relevance**: Responding with content relevant to the user's
------------------------------------------------------------

Category: Factual
User: What is the role of self-attention in Transformers?
Chatb

In [8]:
print("Empty input test:")
print(generate_response(""))

history = [
    {"role": "user", "content": "What is a Transformer?"},
    {
        "role": "assistant",
        "content": "A Transformer is a neural network architecture."
    }
]

print("\nFollow-up conversation test:")
print(
    chatbot_reply(
        "What are its advantages?",
        history
    )
)

Empty input test:
Please enter a valid message.

Follow-up conversation test:
An error occurred: 


In [9]:
requirements = """torch
transformers
accelerate
gradio
pandas
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully.")
print(requirements)

requirements.txt created successfully.
torch
transformers
accelerate
gradio
pandas

